# 📚 Notebook 04 — Feature Engineering
## *Converting complex time-series signals into structured tabular data*

---

**What you'll learn in this notebook:**
- Why machine learning models prefer structured features over raw time-series data
- Handcrafted domain-specific features vs. automatic feature learning
- Shape features: U-shape (planets) vs. V-shape (stellar companions)
- The Odd-Even depth check: detecting binary star systems with unequal light drops
- Integrating stellar metadata (temperature, radius, mass) to assist the classifier
- Structuring a feature matrix for scikit-learn training

**Prerequisites:** Run notebook `03_bls_detection.ipynb` first!


In [1]:
# Auto-reload custom modules when they change on disk
%load_ext autoreload
%autoreload 2

# ============================================================
# Ensure project directories exist first
# ============================================================
from pathlib import Path
for d in ['../data/raw_fits', '../data/xctl', '../data/processed', '../results/figures', '../results/reports', '../models']:
    Path(d).mkdir(parents=True, exist_ok=True)

# ============================================================
# Setup: Import packages and custom modules
# ============================================================
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk

plt.style.use('dark_background')

# Import our custom feature extraction functions
from src.preprocess import load_fits, preprocess_lightcurve
from src.detect import run_bls
from src.features import extract_features, features_to_array, FEATURE_NAMES

print("✅ Setup complete!")

✅ Setup complete!


---
## 📚 Concept 1: What is Feature Engineering?

In classical Machine Learning (like Random Forests or Gradient Boosting), we don't feed the raw, noisy 20,000-point time-series light curve directly into the model. Doing so leads to overfitting, extremely slow training, and poor generalisation.

Instead, we extract **handcrafted features** — a fixed set of meaningful numbers that summarize the signal.

A good feature has three properties:
1. **Informative**: It varies significantly between different classes (e.g., planet transit depth is small, binary depth is large).
2. **Invariant**: It doesn't change based on irrelevant variables (like how bright the host star is).
3. **Robust**: It isn't easily corrupted by minor noise.

Let's load the WASP-121 data, run the BLS transit search, and extract our features!

In [2]:
# 1. Load FITS file and run our preprocessing pipeline
fits_dir = ROOT / "data" / "raw_fits"
fits_files = list(fits_dir.glob("**/*261136679*.fits"))

if fits_files:
    lc = load_fits(fits_files[0])
else:
    lc = lk.search_lightcurve("TIC 261136679", mission="TESS", cadence="short")[0].download()

time, flux, flux_err = preprocess_lightcurve(lc)

# 2. Run BLS to get signal parameters
periods, power, bls_params = run_bls(time, flux, flux_err, period_min=0.5, period_max=10.0, n_periods=10000)

# 3. Extract the features using our custom module
# We will mock some stellar metadata parameters that usually come from the TIC
stellar_metadata = {
    "Teff": 6460.0,       # Stellar Temperature (Kelvin)
    "rad": 1.458,         # Stellar Radius (in Solar Radii)
    "mass": 1.35,         # Stellar Mass (in Solar Masses)
    "Tmag": 10.15,        # Star Brightness
    "contratio": 0.001,   # Aperture contamination
}

features = extract_features(time, flux, flux_err, bls_params, stellar_params=stellar_metadata)

print(f"✅ Successfully extracted {len(features)} features!")

✅ Successfully extracted 25 features!


### Inspecting the Tabular Features
Let's convert our dictionary of features into a pandas DataFrame to look at the numbers. These represent the exact mathematical summary that our Random Forest will use to make its decision!

In [3]:
df_features = pd.DataFrame([features])
pd.set_option('display.max_columns', None)

print("📊 FEATURE TABLE FOR WASP-121b:")
print("=" * 40)
for k, v in sorted(features.items()):
    print(f"  {k:<30} = {v:.6f}")

📊 FEATURE TABLE FOR WASP-121b:
  bls_period                     = 6.268291
  bls_power                      = 33.574529
  bls_snr                        = 5.794353
  contamination                  = 0.001000
  even_depth                     = -0.000005
  mstar                          = 1.350000
  n_transits                     = 5.000000
  odd_depth                      = -0.000025
  odd_even_diff                  = 0.000020
  odd_even_ratio                 = 20.451056
  oot_kurtosis                   = 0.692595
  oot_median                     = 0.999999
  oot_rms                        = 0.000125
  oot_skewness                   = -0.029149
  rstar                          = 1.458000
  secondary_depth                = 0.000006
  secondary_to_primary_ratio     = 6.374157
  shape_curvature                = -0.000017
  teff                           = 6460.000000
  tmag                           = 10.150000
  transit_depth                  = -0.000007
  transit_duration_hrs           =

---
## 📚 Concept 2: Behind the Features — Shape and Symmetry Checks

Let's look at three of the most powerful features we extract:

### 1. Curvature (U-shape vs. V-shape)
- **Planet transits** are **U-shaped**: they have a flat bottom because the planet is fully in front of the star and blocks a constant amount of light during its passage.
- **Eclipsing Binaries** are often **V-shaped**: the eclipse reaches a sharp minimum and immediately starts rising, as the two stars are different sizes or orbit in a grazing eclipse.
- We calculate `shape_curvature` by comparing the flux at the edges of the transit to the flux at the very centre.

### 2. Odd-Even Depth Difference
- Eclipsing binaries consist of two stars orbiting a shared centre of mass. 
- Transit 1: Star A passes in front of Star B.
- Transit 2: Star B passes in front of Star A.
- If the stars have different surface brightnesses, the depths of the odd transits and even transits will be unequal!
- Planets only have ONE transit depth (the planet is dark).
- We calculate `odd_even_diff` to detect these alternating variations.

### 3. Out-of-Transit Scatter (`oot_rms`)
- High noise in the baseline indicates a highly active star (starspots, flares) or a poor quality detection.
- If `bls_snr` is high but `oot_rms` is also very high, the signal might just be a stellar activity flare rather than a planet.

Let's check the shape and odd-even values we computed for WASP-121b:

In [4]:
print("📈 Key Diagnostic Features:")
print(f"   U vs V Curvature:       {features['shape_curvature']:.6f} (positive = flatter bottom/U-shape)")
print(f"   Odd-Even Difference:    {features['odd_even_diff']:.6f} (near 0 = equal transit depths)")
print(f"   Secondary-to-Primary:   {features['secondary_to_primary_ratio']:.6f} (near 0 = no secondary eclipse)")
print(f"   Out-of-Transit RMS:     {features['oot_rms']:.6f} (normal baseline noise level)")

📈 Key Diagnostic Features:
   U vs V Curvature:       -0.000017 (positive = flatter bottom/U-shape)
   Odd-Even Difference:    0.000020 (near 0 = equal transit depths)
   Secondary-to-Primary:   6.374157 (near 0 = no secondary eclipse)
   Out-of-Transit RMS:     0.000125 (normal baseline noise level)


---
## 📚 Concept 3: Preparing the Feature Matrix for scikit-learn

For scikit-learn models, we need to convert our dictionary features into a single, flat 1D numpy array. 
We must ensure that **every star's feature array has the exact same order of columns**.

We use `features_to_array()` which sorts the dictionary keys alphabetically to ensure consistent column placement.

In [5]:
# Convert dictionary to a sorted numpy array
feature_vector = features_to_array(features)

print(f"Flat Feature Array shape: {feature_vector.shape}")
print(f"Feature Array contents:\n{feature_vector}")

# Show the alphabetical column alignment
print("\nFirst 5 sorted feature column names:")
print(sorted(features.keys())[:5])

Flat Feature Array shape: (25,)
Feature Array contents:
[ 6.2682910e+00  3.3574528e+01  5.7943530e+00  1.0000000e-03
 -4.8797297e-06  1.3500000e+00  5.0000000e+00 -2.5330784e-05
  2.0451056e-05  2.0451056e+01  6.9259542e-01  9.9999917e-01
  1.2515376e-04 -2.9149123e-02  1.4579999e+00  6.3741568e-06
  6.3741570e+00 -1.7117849e-05  6.4600000e+03  1.0150000e+01
 -7.4078653e-06  2.4000001e+00 -1.2774128e-05  9.9964863e-01
  1.2863868e-04]

First 5 sorted feature column names:
['bls_period', 'bls_power', 'bls_snr', 'contamination', 'even_depth']


---
## ✅ Notebook 04 Summary

**ML concepts learned:**
- **Dimensionality reduction**: mapping high-dimensional time-series data (20k points) to compact low-dimensional representations (25 features).
- **Domain knowledge injection**: computing metrics based on astrophysics (odd-even depths, secondary eclipses) to guide the classifier.
- **Feature formatting**: arranging dictionary data into structured matrices for scikit-learn models.

**Astronomy concepts learned:**
- Transit shape geometry (U-shape vs. V-shape)
- Odd-even transit alternation in binary stars
- Integrating catalog metadata with photometric light curve data

**Next:** Notebook `05_random_forest.ipynb` — training our first classical Machine Learning classifier (Random Forest) on this tabular dataset!